# Wearable Activity Classification

This notebook sets up an experiment in which LLM agents (students) are tasked
with writing a `solution` function for a **multi-class classification problem**
using wearable sensor summaries from Apple Watch and Fitbit devices. Over
multiple learning cycles, the agents iteratively refine their code based on
performance feedback, e.g., Accuracy, Macro F1-Score, and execution time.
Crucially, participants in the validation set are never seen during training,
testing cross-person physiological generalization. The notebook includes data
preparation, agent configuration, the learning loop, and visualization of
results.

### Running in Colab vs. Local Jupyter
- **Google Colab**: Save your `GEMINI_API_KEY` under Colab **Secrets** (key icon 🔑 on the left sidebar) with notebook access enabled. The **Setup** cell below automatically installs `codegen_classroom` from PyPI if it is not already installed, and `ensure_data()` downloads `aw_fb_data.csv` if not present.
- **Local Jupyter**: If `codegen_classroom` is already installed in your virtual environment, the Setup cell skips installation and imports the package directly.

## Setup


In [ ]:
# @title Install
# Or install latest from GitHub: !pip install -q "codegen-classroom[lightgbm,notebook] @ git+https://github.com/google-research/codegen-classroom.git"
try:
  from codegen_classroom.tasks.wearable import config
except ImportError:
  !pip install -q "codegen-classroom[lightgbm,notebook]"

In [ ]:
# @title Imports
from collections import defaultdict
import functools
import logging
import os

from codegen_classroom import agent_config
from codegen_classroom import agents
from codegen_classroom import data_utils
from codegen_classroom import plotting
from codegen_classroom import reporting
from codegen_classroom import scoring
from codegen_classroom import student_learning
from codegen_classroom.tasks.wearable import config
from codegen_classroom.tasks.wearable import data_loader
from codegen_classroom.tasks.wearable import run_wearable
import pandas as pd

agent_config.load_env()
reporting.suppress_common_warnings()
DARK_MODE = False

## Data


In [ ]:
TASK_DIR = run_wearable.resolve_task_dir()
DATA_DIR = (
    os.path.join(TASK_DIR, "data") if os.access(TASK_DIR, os.W_OK) else "data"
)
LEADERBOARD_PATH = os.path.join(TASK_DIR, "leaderboard.tsv")

if os.path.exists(LEADERBOARD_PATH):
  reporting.render_leaderboard(LEADERBOARD_PATH)

data_loader.ensure_data(DATA_DIR)
X, y, groups = data_loader.load_preprocessed_wearable_data(
    data_dir=DATA_DIR, y_column="activity_encoded"
)

(
    X_train,
    X_val,
    X_test_final,
    y_train,
    y_val,
    y_test_final,
    groups_train,
    groups_val,
    groups_test_final,
    X_learning,
    y_learning,
    groups_learning,
) = data_loader.split_wearable_data(X, y, groups, random_state=config.VAL_SEED)
print(
    f"train={len(X_train)} ({groups_train.nunique()} users) "
    f"val={len(X_val)} ({groups_val.nunique()} users) "
    f"test={len(X_test_final)} ({groups_test_final.nunique()} users)"
)

## Student Instructions


In [ ]:
STUDENT_INSTRUCTIONS = config.build_student_instructions(
    feature_names=list(X.columns),
    train_rows=len(X_train),
    val_rows=len(X_val),
    final_eval_rows=len(X_learning),
    train_users=groups_train.nunique(),
    val_users=groups_val.nunique(),
    final_eval_users=groups_learning.nunique(),
    timeout_s=config.RUN_DEFAULTS["timeout_s"],
    final_eval_timeout_s=config.RUN_DEFAULTS["final_eval_timeout_s"],
    metric_weights=config.DEFAULT_METRIC_WEIGHTS,
)

reporting.render_markdown(STUDENT_INSTRUCTIONS)

## Classroom Setup


In [ ]:
# @title Classroom Setup
EXPERIMENT_NAME = "wearable_notebook"  # @param {type:"string"}
VERBOSE_DEBUG = True  # @param {type:"boolean"}
NUM_STUDENTS = config.RUN_DEFAULTS["num_students"]
NUM_LEARNING_CYCLES = config.RUN_DEFAULTS["num_cycles"]
MODEL_NAMES = [
    config.DEFAULT_STUDENT_MODELS[i % len(config.DEFAULT_STUDENT_MODELS)]
    for i in range(NUM_STUDENTS)
]

RUNS_BASE_PATH = data_utils.experiments_root(config.TASK_NAME)
RUN_ID = data_utils.generate_run_id()
RUN_DIR = os.path.join(RUNS_BASE_PATH, EXPERIMENT_NAME, RUN_ID)
data_utils.create_dir(RUN_DIR)

logger = reporting.setup_logger(
    "codegen_classroom",
    level=logging.DEBUG if VERBOSE_DEBUG else logging.INFO,
    log_file=os.path.join(RUN_DIR, "run.log"),
)

student_cfgs = student_learning.get_classroom_student_configs(
    system_instruction=STUDENT_INSTRUCTIONS,
    model_names=MODEL_NAMES,
    temperature=0.85,
    max_output_tokens=config.RUN_DEFAULTS["max_output_tokens"],
)
students = {
    cfg.name: agents.StudentAgent(config=cfg, run_base_dir=RUN_DIR)
    for cfg in student_cfgs
}

## Learning


In [ ]:
# @title Learning Loop
current_classroom = list(students.values())
results_history = defaultdict(list)
active_metrics = ["overall_score"] + list(config.DEFAULT_METRIC_WEIGHTS.keys())
validator = functools.partial(
    scoring.classification_prediction_validator, expected_columns=1
)

for current_cycle in range(NUM_LEARNING_CYCLES):
  if not current_classroom:
    break
  classroom_feedback = scoring.get_classroom_feedback_string(
      cycle_num=current_cycle,
      current_classroom=current_classroom,
      classroom_results_history=results_history,
      metric_keys=active_metrics,
  )
  classroom_results = student_learning.run_parallel_classroom_learning_cycle(
      current_classroom=current_classroom,
      x_train=X_train,
      y_train=y_train,
      x_val=X_val,
      y_val=y_val,
      current_cycle=current_cycle,
      student_initial_prompt=config.STUDENT_INITIAL_PROMPT,
      student_function_name=config.STUDENT_FUNCTION_NAME,
      code_execution_max_retries=2,
      code_execution_timeout_seconds=config.RUN_DEFAULTS["timeout_s"],
      metric_weights=config.DEFAULT_METRIC_WEIGHTS,
      early_stopping_patience=config.RUN_DEFAULTS["early_stopping_patience"],
      student_refine_instruction_options=config.STUDENT_REFINE_INSTRUCTION_OPTIONS,
      max_mutation_count=config.RUN_DEFAULTS["max_mutation_count"],
      mutation_strategy_options=[
          agents.MutationStrategy.SELF_BEST_CODE,
          agents.MutationStrategy.BEST_PEER_CROSSOVER,
          agents.MutationStrategy.CONTRASTIVE_PEER_CROSSOVER,
      ],
      validator=validator,
      classroom_feedback=classroom_feedback,
      num_workers=len(current_classroom),
      verbose_logging=VERBOSE_DEBUG,
  )
  current_classroom = student_learning.process_cycle_results(
      current_classroom, classroom_results, results_history, current_cycle
  )
  for student in students.values():
    student.reset_history()

plotting.plot_progress(
    os.path.join(RUN_DIR, "students"),
    metric_names=active_metrics,
    dark_mode=DARK_MODE,
)

## Evaluation


In [ ]:
# @title Final Evaluation and Report
FINAL_RESULTS_PATH = os.path.join(RUN_DIR, "final_eval_results.json")

eval_results, _ = student_learning.run_parallel_classroom_final_evaluation(
    students=students,
    x_learning=X_learning,
    y_learning=y_learning,
    x_test_final=X_test_final,
    y_test_final=y_test_final,
    student_function_name=config.STUDENT_FUNCTION_NAME,
    metric_weights=config.DEFAULT_METRIC_WEIGHTS,
    code_execution_timeout_seconds=config.RUN_DEFAULTS["final_eval_timeout_s"],
    enforce_timeout=True,
    strategies=["first", "best", "last"],
    validator=validator,
    num_workers=len(students),
    final_results_path=FINAL_RESULTS_PATH,
    predictions_dir=os.path.join(RUN_DIR, "predictions"),
)

plotting.plot_final_test(
    metrics=active_metrics,
    final_results_path=FINAL_RESULTS_PATH,
    dark_mode=DARK_MODE,
)

sections = reporting.build_run_report(
    RUN_DIR,
    students=list(students.values()),
    metric_weights=config.DEFAULT_METRIC_WEIGHTS,
)
reporting.render_report(sections, dark_mode=DARK_MODE)
reporting.write_report(
    RUN_DIR, sections, metric_weights=config.DEFAULT_METRIC_WEIGHTS
)

## Code Analysis


In [ ]:
# @title Best Solution

# Find the winner from evaluation results
best_student_name, best_eval_data = (
    student_learning.get_best_solution_from_eval_results(
        eval_results,
        metric_key="overall_score",
        solution_strategy="best",
        min_cycle=0,
    )
)

if best_student_name:
  student_csv_path = os.path.join(
      RUN_DIR, "students", f"{best_student_name}.csv"
  )
  if os.path.exists(student_csv_path):
    student_history_df = pd.read_csv(student_csv_path)
    top_cycle_result = student_history_df.sort_values(
        by="overall_score", ascending=False
    ).iloc[0]

    reporting.render_markdown(
        (
            "# Winner: %s\n### Score: %0.4f (Cycle: %d)\n### Plan\n%s\n\n###"
            " Code\n```python\n%s\n```\n\n### Execution Output\n```\n%s\n```"
        )
        % (
            best_student_name,
            top_cycle_result["overall_score"],
            top_cycle_result["cycle"],
            top_cycle_result["plan"],
            top_cycle_result["code"],
            top_cycle_result["execution_feedback"],
        )
    )